# Naive Baseline

Lag-based baseline: predict y[t] = har_ma_125[t] (125-period rolling mean of adjusted RV).

In [ ]:
import subprocess, sys, os

# Clone repo if not present
if not os.path.exists("harxhar"):
    subprocess.run(["git", "clone", "https://github.com/your-org/harxhar.git"], check=True)
os.chdir("harxhar/colab")

# Install dependencies
subprocess.check_call([sys.executable, "-m", "pip", "-q", "install",
                       "numpy", "pandas", "scikit-learn", "pyarrow", "tqdm"])

sys.path.insert(0, ".")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.loading import load_raw_data
from src.transforms import robust_transform
from src.ml_baseline import resolve_har_lags, generate_har_features, apply_horizon_shift, apply_duan_smearing

HORIZON = 1
TRAIN_WINDOW_DAYS = 500
DATA_PATH = "all30min"
PERIODS_PER_DAY = 48
TRAIN_WINDOW = TRAIN_WINDOW_DAYS * PERIODS_PER_DAY

# Load + transform + features
df = load_raw_data(DATA_PATH)
adj_rv, baseline = robust_transform(df, "RV", is_target=True)
df["adj_RV"] = adj_rv
df["baseline"] = baseline
df, feature_names = generate_har_features(df, target_col="adj_RV")

max_lag = resolve_har_lags()[-1]
df = df.iloc[max_lag:].reset_index(drop=True)

X = df[feature_names].values.astype(np.float64)
y = df["adj_RV"].values.astype(np.float64)
dates = df["t"]
baselines = df["baseline"].values.astype(np.float64)
X, y, dates, baselines = apply_horizon_shift(X, y, dates, baselines, HORIZON)

print(f"Data: {X.shape[0]} samples, {len(feature_names)} features")
print(f"Feature names: {feature_names}")

In [ ]:
# Naive model: y_pred[t] = har_ma_125[t]
lag_125_index = feature_names.index("har_ma_125")
print(f"Using feature index {lag_125_index} (har_ma_125) as naive predictor")

oos_start = TRAIN_WINDOW
X_oos = X[oos_start:]
y_oos = y[oos_start:]
dates_oos = dates.iloc[oos_start:].values
baselines_oos = baselines[oos_start:]
preds = X_oos[:, lag_125_index]

# Autocorrelation of adjusted RV
from pandas.plotting import autocorrelation_plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
autocorrelation_plot(pd.Series(y_oos[:5000]), ax=axes[0])
axes[0].set_title("Autocorrelation of adj_RV")
axes[0].set_xlim(0, 200)
axes[1].scatter(preds[:2000], y_oos[:2000], alpha=0.3, s=5)
axes[1].set_xlabel("Predicted (har_ma_125)")
axes[1].set_ylabel("Actual (adj_RV)")
axes[1].set_title("Naive Baseline: Predicted vs Actual")
plt.tight_layout()
plt.show()

pred_raw, true_raw = apply_duan_smearing(preds, y_oos, baselines_oos)

In [ ]:
from src.evaluation import calculate_metrics

results_df = pd.DataFrame({
    "date": dates_oos,
    "horizon": HORIZON,
    "true_adj": y_oos,
    "pred_adj": preds,
    "true_raw": true_raw,
    "pred_raw": pred_raw,
})

metrics = calculate_metrics(results_df)
print("\n".join(f"{k:>10s}: {v:.6f}" for k, v in metrics.items()))

In [ ]:
%%writefile ../src/ml_baseline.py
"""Naive lag-based baseline volatility forecast executor.

Predicts y[t] = har_ma_125[t] (the 125-period HAR rolling mean).
Same CLI interface as ml_ridge.py.  No imports from core/ or projects/.
"""

import argparse
import os

import numpy as np
import pandas as pd

from src.loading import load_raw_data
from src.transforms import robust_transform

# ── Constants ─────────────────────────────────────────────────────────────
PERIODS_PER_DAY = 48


# ── HAR lag features (same as ml_ridge) ───────────────────────────────────

def resolve_har_lags(max_lag: int = 3125) -> list[int]:
    seq, v = [], 1
    while v <= max_lag:
        seq.append(v)
        v *= 5
    return seq


def generate_har_features(
    df: pd.DataFrame, target_col: str = "adj_RV"
) -> tuple[pd.DataFrame, list[str]]:
    lags = resolve_har_lags()
    features: dict[str, pd.Series] = {}
    feature_names: list[str] = []
    for lag in lags:
        name = f"har_ma_{lag}"
        features[name] = (
            df[target_col].rolling(window=lag, min_periods=1).mean().shift(1)
        )
        feature_names.append(name)
    feat_df = pd.DataFrame(features, index=df.index)
    return pd.concat([df, feat_df], axis=1), feature_names


# ── Horizon shift ─────────────────────────────────────────────────────────

def apply_horizon_shift(
    X: np.ndarray,
    y: np.ndarray,
    dates: pd.Series,
    baselines: np.ndarray,
    horizon: int,
) -> tuple[np.ndarray, np.ndarray, pd.Series, np.ndarray]:
    if horizon <= 1:
        return X, y, dates, baselines
    shift = horizon - 1
    return (
        X[:-shift],
        y[shift:],
        dates.iloc[:-shift].reset_index(drop=True),
        baselines[shift:],
    )


# ── Duan smearing (inline) ───────────────────────────────────────────────

def apply_duan_smearing(
    forecasts: np.ndarray, y_true: np.ndarray, baselines: np.ndarray
) -> tuple[np.ndarray, np.ndarray]:
    smear = np.mean((y_true - forecasts) ** 2)
    pred_raw = (forecasts**2 + smear) * baselines
    true_raw = (y_true**2) * baselines
    return pred_raw, true_raw


# ── CLI ───────────────────────────────────────────────────────────────────

def main() -> None:
    parser = argparse.ArgumentParser(description="Naive baseline backtest")
    parser.add_argument("--data-path", default="all30min")
    parser.add_argument("--horizon", type=int, default=1)
    parser.add_argument(
        "--train-window", type=int, default=500, help="training window in days (burn-in)"
    )
    parser.add_argument("--start", type=int, default=0)
    parser.add_argument("--end", type=int, default=-1)
    parser.add_argument("--output-file", required=True)
    args = parser.parse_args()

    train_win_periods = args.train_window * PERIODS_PER_DAY

    # 1. Load data
    df = load_raw_data(args.data_path)

    # 2. Robust transform on RV
    adj_rv, baseline = robust_transform(df, "RV", is_target=True)
    df["adj_RV"] = adj_rv
    df["baseline"] = baseline

    # 3. HAR features
    df, feature_names = generate_har_features(df, target_col="adj_RV")

    # 4. Drop initial NaN rows
    max_lag = resolve_har_lags()[-1]
    df = df.iloc[max_lag:].reset_index(drop=True)

    # 5. Extract numpy arrays
    X = df[feature_names].values.astype(np.float64)
    y = df["adj_RV"].values.astype(np.float64)
    dates = df["t"]
    baselines = df["baseline"].values.astype(np.float64)

    # 6. Horizon shift
    X, y, dates, baselines = apply_horizon_shift(X, y, dates, baselines, args.horizon)

    # 7. Slice
    start = args.start
    end = len(X) if args.end == -1 else args.end

    X_chunk = X[start:end]
    y_chunk = y[start:end]
    dates_chunk = dates.iloc[start:end].reset_index(drop=True)
    baselines_chunk = baselines[start:end]

    if train_win_periods >= len(X_chunk):
        raise ValueError(
            f"train_window ({train_win_periods} periods) >= chunk size ({len(X_chunk)})"
        )

    # 8. Naive prediction: y_pred[t] = har_ma_125[t]
    lag_125_index = feature_names.index("har_ma_125")

    oos_start = train_win_periods
    X_oos = X_chunk[oos_start:]
    y_oos = y_chunk[oos_start:]
    dates_oos = dates_chunk.iloc[oos_start:].values
    baselines_oos = baselines_chunk[oos_start:]

    preds = X_oos[:, lag_125_index]

    # 9. Duan smearing + save
    pred_raw, true_raw = apply_duan_smearing(preds, y_oos, baselines_oos)

    results = pd.DataFrame(
        {
            "date": dates_oos,
            "horizon": args.horizon,
            "true_adj": y_oos,
            "pred_adj": preds,
            "true_raw": true_raw,
            "pred_raw": pred_raw,
        }
    )

    os.makedirs(os.path.dirname(args.output_file) or ".", exist_ok=True)
    results.to_csv(args.output_file, index=False)
    print(f"Saved {len(results)} rows → {args.output_file}")


if __name__ == "__main__":
    main()